## Update Notes For New CSV Datasets (pheno_crop_v2)

This notebook was updated to support the new CSV schema in `ground_truth.csv`, `sentinel_1/*.csv`, and `sentinel_2/*.csv`.

### What changed
- Ground truth labels now come from `stage_name` (text labels) instead of `stage_code`.
- Added a deterministic label encoder in-notebook: `stage_to_idx` / `idx_to_stage`.
- Training, dataset class, evaluation, and export now use dynamic `num_classes` from the data.
- Class weights are computed from the detected class count (no hard-coded 5).
- Evaluation now prints class names and supports any number of classes.
- Prediction export now writes probability columns dynamically (`prob_stage_0 ... prob_stage_n`).
- The dataset exploration cells were kept before the model-building section so schema checks happen first.

### Confirmed new dataset schema used
- `ground_truth.csv`: `plot_id`, `date`, `stage_name`, `NDVI`, `stress`, `yield_est`, `confidence`, `raw_coordinates`
- Sentinel-1 CSV columns: `date`, `VV`, `VH`, `VH_VV_Ratio`, `orbit`
- Sentinel-2 CSV columns: `date`, `cloud_pct`, `NDVI`, `NDWI`, `NDRE`, `EVI`, `SAVI`, `MSAVI`, `NDMI`, `GNDVI`

## Setup

The Colab mount/check cell below is optional.
- Run it only if you are on Google Colab and want to mount Google Drive.
- If you are local, skip it and go directly to Cell 4.

Set `DATASET_ROOT` in Cell 4 to the directory that directly contains:

```bash
ground_truth.csv
sentinel_1/
sentinel_2/
```

Examples:
- Colab: `/content/drive/MyDrive/pheno_crop_v2`
- Local repo clone: `/home/<user>/.../pheno_crop/dataset/pheno_crop_v2`

In [2]:
import os
from collections import Counter

USE_COLAB_MOUNT = True  # Set True only when running on Google Colab

# new updated path to the second version of the dataset on Google Drive
COLAB_ROOT = "/content/drive/MyDrive/pheno_crop_v2"

if USE_COLAB_MOUNT:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f"Colab mount skipped: {exc}")


def verify_dataset(base_paths):
    print("VERIFYING DATASET STRUCTURE...")
    print("=" * 60)

    total_dataset_size = 0
    total_dataset_files = 0

    for base_path in base_paths:
        if not os.path.exists(base_path):
            print(f"Not found: {base_path} (Skipping)")
            continue

        for root, dirs, files in os.walk(base_path):
            folder_size = sum(os.path.getsize(os.path.join(root, f)) for f in files)
            total_dataset_size += folder_size
            total_dataset_files += len(files)

            ext_counts = Counter(os.path.splitext(f)[1].lower() for f in files)

            level = root.replace(base_path, '').count(os.sep)
            indent = ' ' * 4 * level
            folder_name = os.path.basename(root) if root != base_path else os.path.basename(base_path)

            print(f"{indent} {folder_name}/  [{len(files)} files | {folder_size / (1024*1024):.2f} MB]")

            if files:
                for ext, count in ext_counts.items():
                    ext_name = ext if ext else "no_extension"
                    print(f"{indent}    ├── {count} {ext_name} files")

                examples = sorted(files)
                if len(files) >= 3:
                    examples_str = f"{examples[0]}, {examples[1]}, {examples[2]}, ..."
                else:
                    examples_str = ", ".join(examples)
                print(f"{indent}    └── Examples: {examples_str}\n")
            else:
                print(f"{indent}    └── (Empty folder)\n")

    print("=" * 60)
    print(f"Total Dataset Files: {total_dataset_files}")
    print(f"Total Dataset Size:  {total_dataset_size / (1024*1024):.2f} MB")

paths_to_check = [COLAB_ROOT]

if any(os.path.exists(p) for p in paths_to_check):
    verify_dataset(paths_to_check)
else:
    print("Dataset verification skipped. Set USE_COLAB_MOUNT=True on Colab if needed.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
VERIFYING DATASET STRUCTURE...
 pheno_crop_v2/  [1 files | 5.23 MB]
    ├── 1 .csv files
    └── Examples: ground_truth.csv

     sentinel_2/  [176 files | 3.05 MB]
        ├── 176 .csv files
        └── Examples: plot_100_indices.csv, plot_101_indices.csv, plot_102_indices.csv, ...

     sentinel_1/  [176 files | 0.52 MB]
        ├── 176 .csv files
        └── Examples: plot_100_sar.csv, plot_101_sar.csv, plot_102_sar.csv, ...

Total Dataset Files: 353
Total Dataset Size:  8.81 MB


In [11]:
from pathlib import Path
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ===================== USER CONFIG =====================
# Set this to the directory that directly contains:
# - ground_truth.csv
# - sentinel_1/
# - sentinel_2/
DATASET_ROOT = "/content/drive/MyDrive/pheno_crop_v2"

# Optional local fallback (used automatically if DATASET_ROOT does not exist)
LOCAL_DATASET_ROOT = Path("./dataset/pheno_crop_v2")

# Where model and prediction files are saved.
OUTPUT_DIR = "./models"
# =======================================================

dataset_root = Path(DATASET_ROOT).expanduser()
if not dataset_root.exists() and LOCAL_DATASET_ROOT.exists():
    dataset_root = LOCAL_DATASET_ROOT.resolve()
    print(f"DATASET_ROOT not found. Using local fallback: {dataset_root}")

s1_dir = dataset_root / "sentinel_1"
s2_dir = dataset_root / "sentinel_2"
gt_path = dataset_root / "ground_truth.csv"
output_dir = Path(OUTPUT_DIR).expanduser()

print(f"Dataset root: {dataset_root}")
print(f"Ground truth file exists: {gt_path.exists()}")
print(f"S1 dir exists: {s1_dir.exists()}")
print(f"S2 dir exists: {s2_dir.exists()}")
print(f"Output dir: {output_dir}")

if not gt_path.exists():
    print("Warning: update DATASET_ROOT to the folder that contains ground_truth.csv.")

Dataset root: /content/drive/MyDrive/pheno_crop_v2
Ground truth file exists: True
S1 dir exists: True
S2 dir exists: True
Output dir: models


In [7]:
sar1_path = s1_dir / "plot_100_sar.csv"
sar1 = pd.read_csv(sar1_path)
print(f"SAR1 shape: {sar1.shape}")
sar1.head(5)

SAR1 shape: (38, 5)


,date,VV,VH,VH_VV_Ratio,orbit
0,2025-11-02,-11.114252,-17.785049,-6.670797,DESCENDING
1,2025-11-06,-12.370057,-18.599606,-6.229549,ASCENDING
2,2025-11-11,-13.196421,-19.246956,-6.050535,ASCENDING
3,2025-11-14,-11.860846,-17.750793,-5.889947,DESCENDING
4,2025-11-18,-12.177608,-15.545235,-3.367627,ASCENDING


In [10]:
sen2_path = s2_dir / "plot_100_indices.csv"
sen2 = pd.read_csv(sen2_path)
print(f"Sentinel-2 shape: {sen2.shape}")
sen2.head(5)

Sentinel-2 shape: (103, 10)


,date,cloud_pct,NDVI,NDWI,NDRE,EVI,SAVI,MSAVI,NDMI,GNDVI
0,2025-11-05,0.034743,0.696170,-0.628232,0.453282,0.473258,0.434924,0.418866,0.323619,0.628232
1,2025-11-05,0.035494,0.695604,-0.627801,0.452968,0.473817,0.434931,0.418892,0.323266,0.627801
2,2025-11-07,0.025121,0.753805,-0.673967,0.519413,0.522705,0.465360,0.455406,0.311954,0.673967
3,2025-11-07,0.026617,0.753816,-0.673733,0.518307,0.523655,0.465767,0.455933,0.312968,0.673733
4,2025-11-07,0.017220,0.796328,-0.724102,0.548084,0.486202,0.467333,0.456946,0.328422,0.724102


In [12]:
gt = pd.read_csv(gt_path)
print(f"Ground truth shape: {gt.shape}")
gt.head(5)

Ground truth shape: (23460, 8)


,plot_id,date,stage_name,NDVI,stress,yield_est,confidence,raw_coordinates
0,10,2025-12-01,Bare,0.229508,Normal,0.074494,0.621642,"[[[84.83436115, 25.53779848], [84.83425789, 25..."
1,10,2025-12-02,Bare,0.228587,Normal,0.074494,0.620948,"[[[84.83436115, 25.53779848], [84.83425789, 25..."
2,10,2025-12-03,Bare,0.198911,Normal,0.074494,0.598710,"[[[84.83436115, 25.53779848], [84.83425789, 25..."
3,10,2025-12-04,Bare,0.193170,Normal,0.074494,0.598226,"[[[84.83436115, 25.53779848], [84.83425789, 25..."
4,10,2025-12-05,Bare,0.211499,Normal,0.074494,0.605815,"[[[84.83436115, 25.53779848], [84.83425789, 25..."


In [13]:
# Inspect available growth stages from the new ground-truth schema
stage_names = sorted(gt['stage_name'].dropna().unique().tolist())
print(f"Unique stage names ({len(stage_names)}): {stage_names}")

Unique stage names (5): ['Bare', 'Growth', 'Ripening', 'Seedling', 'Tillering']


In [14]:
gt = pd.read_csv(gt_path)
gt["date"] = pd.to_datetime(gt["date"])

# Build stable integer labels from stage_name
stage_names = sorted(gt["stage_name"].dropna().unique().tolist())
stage_to_idx = {name: idx for idx, name in enumerate(stage_names)}
idx_to_stage = {idx: name for name, idx in stage_to_idx.items()}
gt["stage_idx"] = gt["stage_name"].map(stage_to_idx)
num_classes = len(stage_names)

print("Ground-truth rows:", len(gt))
print("Unique plots:", gt["plot_id"].nunique())
print("Detected classes:", num_classes)
print("Stage mapping:", stage_to_idx)
print("Stage distribution (name):")
print(gt["stage_name"].value_counts().sort_index())
print("Stage distribution (idx):")
print(gt["stage_idx"].value_counts().sort_index())

# Plot-level leakage-aware split: train/val/test by plot_id
all_plots = sorted(gt["plot_id"].unique())
plot_df = pd.DataFrame({"plot_id": all_plots})

# First split: hold out test plots
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
idx_trainval, idx_test = next(gss_test.split(plot_df, groups=plot_df["plot_id"]))
trainval_plots = set(plot_df.iloc[idx_trainval]["plot_id"].tolist())
test_plots = set(plot_df.iloc[idx_test]["plot_id"].tolist())

# Second split: split trainval into train/val
trainval_df = plot_df.iloc[idx_trainval].reset_index(drop=True)
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=SEED)  # ~15% of total
idx_train, idx_val = next(gss_val.split(trainval_df, groups=trainval_df["plot_id"]))
train_plots = set(trainval_df.iloc[idx_train]["plot_id"].tolist())
val_plots = set(trainval_df.iloc[idx_val]["plot_id"].tolist())

# Leakage check
assert train_plots.isdisjoint(val_plots)
assert train_plots.isdisjoint(test_plots)
assert val_plots.isdisjoint(test_plots)

print(f"Train plots: {len(train_plots)} | Val plots: {len(val_plots)} | Test plots: {len(test_plots)}")

train_rows = gt[gt["plot_id"].isin(train_plots)]
val_rows = gt[gt["plot_id"].isin(val_plots)]
test_rows = gt[gt["plot_id"].isin(test_plots)]

print(f"Train rows: {len(train_rows)} | Val rows: {len(val_rows)} | Test rows: {len(test_rows)}")

Ground-truth rows: 23460
Unique plots: 204
Detected classes: 5
Stage mapping: {'Bare': 0, 'Growth': 1, 'Ripening': 2, 'Seedling': 3, 'Tillering': 4}
Stage distribution (name):
stage_name
Bare         2360
Growth       5169
Ripening     3682
Seedling     5007
Tillering    7242
Name: count, dtype: int64
Stage distribution (idx):
stage_idx
0    2360
1    5169
2    3682
3    5007
4    7242
Name: count, dtype: int64
Train plots: 142 | Val plots: 31 | Test plots: 31
Train rows: 16330 | Val rows: 3565 | Test rows: 3565


## Model 2: Leakage-Aware Phenology Training

This section trains the leakage-aware model after dataset/schema validation.

Improvements retained from Model 1:
- plot-level group splits to reduce leakage
- strict unseen test split
- macro-F1 and per-class recall tracking
- regularization and early stopping

New dataset alignment:
- labels are encoded from `stage_name`
- class count is detected dynamically from `ground_truth.csv`

### Dataset Preparation

In [15]:
class PhenologyDatasetV2(Dataset):
    def __init__(self, gt_df, s1_dir, s2_dir, lookback_days=90, max_seq_len=30):
        self.gt = gt_df.sort_values(["plot_id", "date"]).reset_index(drop=True)
        self.lookback_days = lookback_days
        self.max_seq_len = max_seq_len
        self.s1_dir = Path(s1_dir)
        self.s2_dir = Path(s2_dir)

        self.s1_cache = {}
        self.s2_cache = {}

        unique_plots = sorted(self.gt["plot_id"].unique())
        for pid in unique_plots:
            s1_path = self.s1_dir / f"plot_{pid}_sar.csv"
            s2_path = self.s2_dir / f"plot_{pid}_indices.csv"

            if s1_path.exists():
                df = pd.read_csv(s1_path)
                df["date"] = pd.to_datetime(df["date"])
                self.s1_cache[pid] = df

            if s2_path.exists():
                df = pd.read_csv(s2_path)
                df["date"] = pd.to_datetime(df["date"])
                self.s2_cache[pid] = df

    def __len__(self):
        return len(self.gt)

    def _build_sequence(self, df, target_date, columns):
        seq = torch.zeros((self.max_seq_len, len(columns)), dtype=torch.float32)
        days = torch.zeros((self.max_seq_len,), dtype=torch.long)
        mask = torch.zeros((self.max_seq_len,), dtype=torch.bool)

        if df is None or df.empty:
            return seq, days, mask

        # Fill any missing columns with zeros to keep model input stable
        for c in columns:
            if c not in df.columns:
                df[c] = 0.0

        start_date = target_date - pd.Timedelta(days=self.lookback_days)
        valid = df[(df["date"] > start_date) & (df["date"] <= target_date)].sort_values("date")

        if valid.empty:
            return seq, days, mask

        feats = valid[columns].fillna(0.0).to_numpy(dtype=np.float32)
        days_ago = (target_date - valid["date"]).dt.days.to_numpy(dtype=np.int64)

        seq_len = min(len(valid), self.max_seq_len)
        seq[-seq_len:] = torch.tensor(feats[-seq_len:], dtype=torch.float32)
        days[-seq_len:] = torch.tensor(days_ago[-seq_len:], dtype=torch.long)
        mask[-seq_len:] = True

        return seq, days, mask

    def __getitem__(self, idx):
        row = self.gt.iloc[idx]
        pid = int(row["plot_id"])
        target_date = row["date"]
        label = int(row["stage_idx"])

        s1_df = self.s1_cache.get(pid)
        s2_df = self.s2_cache.get(pid)

        s1_seq, s1_days, s1_mask = self._build_sequence(s1_df, target_date, ["VV", "VH", "VH_VV_Ratio"])
        s2_seq, s2_days, s2_mask = self._build_sequence(
            s2_df,
            target_date,
            ["NDVI", "NDWI", "NDRE", "EVI", "SAVI", "MSAVI", "NDMI", "GNDVI", "cloud_pct"],
        )

        return {
            "plot_id": pid,
            "date": target_date.strftime("%Y-%m-%d"),
            "s1_feats": s1_seq,
            "s1_days": s1_days,
            "s1_mask": s1_mask,
            "s2_feats": s2_seq,
            "s2_days": s2_days,
            "s2_mask": s2_mask,
            "label": torch.tensor(label, dtype=torch.long),
        }


train_ds = PhenologyDatasetV2(train_rows, s1_dir, s2_dir)
val_ds = PhenologyDatasetV2(val_rows, s1_dir, s2_dir)
test_ds = PhenologyDatasetV2(test_rows, s1_dir, s2_dir)

print(f"Train/Val/Test events: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}")

Train/Val/Test events: 16330 / 3565 / 3565


### Model Setup

In [18]:
class OpticalBranchV2(nn.Module):
    def __init__(self, in_features=9, embed_dim=64, num_layers=2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_features, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(64, embed_dim, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.time_embed = nn.Embedding(120, embed_dim)
        enc = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=4,
            dim_feedforward=128,
            dropout=0.25,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers)

    def forward(self, x, days_ago, mask):
        # x: [B, T, F], mask: [B, T] True for valid steps
        feats = self.cnn(x.transpose(1, 2)).transpose(1, 2)
        days = torch.clamp(days_ago, 0, 119)
        feats = feats + self.time_embed(days)

        # Transformer expects key padding mask True on padded positions
        key_padding_mask = ~mask
        out = self.transformer(feats, src_key_padding_mask=key_padding_mask)

        # Masked mean pooling
        m = mask.unsqueeze(-1).float()
        pooled = (out * m).sum(dim=1) / torch.clamp(m.sum(dim=1), min=1.0)
        return pooled


class RadarBranchV2(nn.Module):
    def __init__(self, in_features=3, embed_dim=64):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_features, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(32, embed_dim, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.gru = nn.GRU(embed_dim, embed_dim, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
        self.proj = nn.Linear(embed_dim * 2, embed_dim)

    def forward(self, x, mask):
        feats = self.cnn(x.transpose(1, 2)).transpose(1, 2)
        out, _ = self.gru(feats)

        m = mask.unsqueeze(-1).float()
        pooled = (out * m).sum(dim=1) / torch.clamp(m.sum(dim=1), min=1.0)
        return self.proj(pooled)


class MasterPhenologyNetV2(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.optical = OpticalBranchV2(in_features=9, embed_dim=64, num_layers=2)
        self.radar = RadarBranchV2(in_features=3, embed_dim=64)
        self.classifier = nn.Sequential(
            nn.Linear(128, 96),
            nn.BatchNorm1d(96),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(96, num_classes),
        )

    def forward(self, s1_feats, s1_mask, s2_feats, s2_days, s2_mask):
        r = self.radar(s1_feats, s1_mask)
        o = self.optical(s2_feats, s2_days, s2_mask)
        fused = torch.cat([r, o], dim=1)
        return self.classifier(fused)


model = MasterPhenologyNetV2(num_classes=num_classes)
print(f"Model V2 ready for {num_classes} classes")

Model V2 ready for 5 classes


In [19]:
def build_loaders(train_ds, val_ds, test_ds, batch_size=32):
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


def compute_class_weights(dataset, num_classes):
    labels = [dataset[i]["label"].item() for i in range(len(dataset))]
    counts = torch.bincount(torch.tensor(labels), minlength=num_classes)
    counts = torch.where(counts == 0, torch.tensor(1), counts)
    weights = len(labels) / (num_classes * counts.float())
    weights = torch.clamp(weights, max=10.0) # prevent insane multipliers
    return weights


def run_epoch(model, loader, criterion, optimizer, device):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_true, all_pred = [], []

    with torch.set_grad_enabled(is_train):
        for batch in loader:
            s1_feats = batch["s1_feats"].to(device)
            s1_mask = batch["s1_mask"].to(device)
            s2_feats = batch["s2_feats"].to(device)
            s2_days = batch["s2_days"].to(device)
            s2_mask = batch["s2_mask"].to(device)
            labels = batch["label"].to(device)

            logits = model(s1_feats, s1_mask, s2_feats, s2_days, s2_mask)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            preds = torch.argmax(logits, dim=1)
            all_true.extend(labels.detach().cpu().numpy().tolist())
            all_pred.extend(preds.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(len(all_true), 1)
    acc = accuracy_score(all_true, all_pred)
    macro_f1 = f1_score(all_true, all_pred, average="macro", zero_division=0)
    return avg_loss, acc, macro_f1, all_true, all_pred


def train_model_v2(model, train_loader, val_loader, train_ds, num_classes, epochs=80, lr=1e-3, patience=12):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    class_weights = compute_class_weights(train_ds, num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)

    best_state = None
    best_val_f1 = -1.0
    wait = 0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc, tr_f1, _, _ = run_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc, va_f1, _, _ = run_epoch(model, val_loader, criterion, None, device)
        scheduler.step(va_f1)

        if va_f1 > best_val_f1:
            best_val_f1 = va_f1
            wait = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Train Loss {tr_loss:.4f} Acc {tr_acc*100:.2f}% F1 {tr_f1:.4f} | "
                f"Val Loss {va_loss:.4f} Acc {va_acc*100:.2f}% F1 {va_f1:.4f}"
            )

        if wait >= patience:
            print(f"Early stopping at epoch {epoch}. Best val macro-F1: {best_val_f1:.4f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, device

### Training

> Note: You get a suggestion when running this cell to use jagged tensors. Do not do that, the API will change and might break the code later on.

In [ ]:
train_loader, val_loader, test_loader = build_loaders(train_ds, val_ds, test_ds, batch_size=32)
trained_model, device = train_model_v2(
    model,
    train_loader,
    val_loader,
    train_ds,
    num_classes=num_classes,
    epochs=100,
    lr=1e-3,
    patience=15,
)

### Evaluation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_and_report(model, loader, split_name, device, num_classes, idx_to_stage):
    criterion = nn.CrossEntropyLoss()
    loss, acc, macro_f1, y_true, y_pred = run_epoch(model, loader, criterion, None, device)

    class_labels = list(range(num_classes))
    class_names = [idx_to_stage[i] for i in class_labels]
    recalls = recall_score(y_true, y_pred, average=None, labels=class_labels, zero_division=0)

    print(f"\n[{split_name}] Loss: {loss:.4f} | Accuracy: {acc*100:.2f}% | Macro-F1: {macro_f1:.4f}")
    print("Per-class recall:")
    for idx, name, rec in zip(class_labels, class_names, recalls):
        print(f"  [{idx}] {name}: {rec:.4f}")

    print(
        classification_report(
            y_true,
            y_pred,
            labels=class_labels,
            target_names=class_names,
            zero_division=0,
        )
    )

    cm = confusion_matrix(y_true, y_pred, labels=class_labels)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", square=True, xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{split_name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Ground Truth")
    plt.tight_layout()
    plt.show()

    return {
        "loss": loss,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "recall": recalls,
        "y_true": y_true,
        "y_pred": y_pred,
    }


val_metrics = evaluate_and_report(trained_model, val_loader, "Validation", device, num_classes, idx_to_stage)
test_metrics = evaluate_and_report(trained_model, test_loader, "Test (Unseen Plots)", device, num_classes, idx_to_stage)

# Save model + test predictions
models_dir = output_dir
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "model_v2.pth"
torch.save({"model_state_dict": trained_model.state_dict(), "seed": SEED, "stage_to_idx": stage_to_idx}, model_path)
print(f"Saved model to: {model_path}")

# Flatten test predictions with metadata
rows = []
trained_model.eval()
with torch.no_grad():
    for batch in test_loader:
        s1_feats = batch["s1_feats"].to(device)
        s1_mask = batch["s1_mask"].to(device)
        s2_feats = batch["s2_feats"].to(device)
        s2_days = batch["s2_days"].to(device)
        s2_mask = batch["s2_mask"].to(device)
        labels = batch["label"].cpu().numpy()

        logits = trained_model(s1_feats, s1_mask, s2_feats, s2_days, s2_mask)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)

        plot_ids = batch["plot_id"].cpu().numpy() if torch.is_tensor(batch["plot_id"]) else batch["plot_id"]
        dates = batch["date"]

        for i in range(len(preds)):
            date_val = dates[i]
            if isinstance(date_val, pd.Timestamp):
                date_val = date_val.strftime("%Y-%m-%d")
            elif hasattr(date_val, "isoformat"):
                date_val = date_val.isoformat()

            row = {
                "plot_id": int(plot_ids[i]),
                "date": str(date_val),
                "actual_stage_idx": int(labels[i]),
                "actual_stage_name": idx_to_stage[int(labels[i])],
                "predicted_stage_idx": int(preds[i]),
                "predicted_stage_name": idx_to_stage[int(preds[i])],
                "correct_prediction": bool(labels[i] == preds[i]),
            }

            for cls_idx in range(num_classes):
                row[f"prob_stage_{cls_idx}"] = round(float(probs[i, cls_idx]) * 100, 3)

            rows.append(row)

pred_path = models_dir / "pred_v2.csv"
pd.DataFrame(rows).to_csv(pred_path, index=False)
print(f"Saved predictions to: {pred_path}")